In [1]:
import os, gc, glob, warnings
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from collections import defaultdict
warnings.filterwarnings('ignore')

BD_FINAL        = 7
K1_FINAL        = 1.0
K2_FINAL        = 2.0
K3_FINAL        = 3.5
MIN_PERSISTENCE = 4
LABEL_LOGIC     = 'OR'

GUARDRAILS = {
    'network.latency': {'media': 0.250, 'alta': 0.350},
    'network.loss':    {'media': 3.0,   'alta': 8.0},
    'network.jitter':  {'media': 7.0,   'alta': 10.0},
}
MIN_MAD = {
    'network.latency': 0.005,
    'network.loss':    0.5,
    'network.jitter':  0.1,
}

START_DATE  = pd.Timestamp('2026-05-01', tz='UTC')
BD_END      = START_DATE + pd.Timedelta(days=BD_FINAL)

SPLIT_DATES = {
    'train': (START_DATE,                           pd.Timestamp('2026-06-20', tz='UTC')),
    'val':   (pd.Timestamp('2026-06-20', tz='UTC'), pd.Timestamp('2026-07-03', tz='UTC')),
    'test':  (pd.Timestamp('2026-07-03', tz='UTC'), pd.Timestamp('2026-07-23', tz='UTC')),
}

WINDOW_MIN  = 30
SAMPLES_W   = 6
METRICS_NET = ['network.latency', 'network.loss', 'network.jitter']

SAVE_DIR  = '/srv/work'
ARCHIVOS  = sorted(
    glob.glob(f'{SAVE_DIR}/te_*_raw_may_w*.parquet') +
    glob.glob(f'{SAVE_DIR}/te_*_raw_w[0-9].parquet')
)
OUT_LABEL = f'{SAVE_DIR}/te_labeled_ventanas_v3.parquet'
OUT_BASE  = f'{SAVE_DIR}/te_baseline_stats_v3.parquet'

print(f"Archivos   : {len(ARCHIVOS)}  (esperado: 39 = 30 train/val + 9 test w6/w7/w8)")
print(f"Baseline   : {START_DATE.date()} -> {BD_END.date()} ({BD_FINAL}d)")
print(f"Eval desde : {BD_END.date()}")
print(f"Salida     : {OUT_LABEL}")
print(f"\nPARAMETROS v7: BD={BD_FINAL}d  K=({K1_FINAL},{K2_FINAL},{K3_FINAL})  pers>={MIN_PERSISTENCE}  logic={LABEL_LOGIC}")
print(f"\nSPLIT_DATES:")
for split, (s, e) in SPLIT_DATES.items():
    print(f"  {split:5s}: {s.date()} -> {e.date()}")


Archivos   : 39  (esperado: 39 = 30 train/val + 9 test w6/w7/w8)
Baseline   : 2026-05-01 -> 2026-05-08 (7d)
Eval desde : 2026-05-08
Salida     : /srv/work/te_labeled_ventanas_v3.parquet

PARAMETROS v7: BD=7d  K=(1.0,2.0,3.5)  pers>=4  logic=OR

SPLIT_DATES:
  train: 2026-05-01 -> 2026-06-20
  val  : 2026-06-20 -> 2026-07-03
  test : 2026-07-03 -> 2026-07-23


In [2]:
PROVIDER_PREFIXES = ['aws', 'azure', 'gcp']
route_keys = set()

for prov in PROVIDER_PREFIXES:
    prov_files = [f for f in ARCHIVOS if f'/te_{prov}_' in f.lower()]
    if not prov_files:
        print(f"⚠️  Sin archivos para {prov}")
        continue
    for f in prov_files:
        tbl = pq.read_table(f, columns=['source_agent','cloud_provider','region'])
        for row in tbl.to_pandas().drop_duplicates().itertuples():
            route_keys.add((row.source_agent, row.cloud_provider, row.region))
        del tbl; gc.collect()
    prov_count = sum(1 for r in route_keys if r[1].lower() == prov)
    print(f"  {prov.upper()}: {prov_count} rutas")

routes_df = pd.DataFrame(list(route_keys), columns=['source_agent','cloud_provider','region'])
routes_df = routes_df.sort_values(['cloud_provider','region','source_agent']).reset_index(drop=True)
print(f"\nRutas totales: {len(routes_df):,}")
print(routes_df.groupby('cloud_provider').size())


  AWS: 414 rutas
  AZURE: 322 rutas
  GCP: 322 rutas

Rutas totales: 1,058
cloud_provider
AWS      414
Azure    322
GCP      322
dtype: int64


In [3]:
PROVIDER_MAP = {
    'aws':   [f for f in ARCHIVOS if '/te_aws_'   in f.lower()],
    'azure': [f for f in ARCHIVOS if '/te_azure_' in f.lower()],
    'gcp':   [f for f in ARCHIVOS if '/te_gcp_'   in f.lower()],
}

baseline_records = []

for prov, files in PROVIDER_MAP.items():
    print(f"[{prov.upper()}] {len(files)} archivos...")
    prov_chunks = []

    for f in files:
        tbl = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric','value'])
        df  = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        mask = (df['_time'] >= START_DATE) & (df['_time'] < BD_END) & (df['metric'].isin(METRICS_NET))
        chunk = df[mask].copy(); del df; gc.collect()
        if len(chunk) > 0:
            prov_chunks.append(chunk)

    if not prov_chunks:
        print(f"  ⚠️  Sin datos baseline para {prov}")
        continue

    prov_df = pd.concat(prov_chunks, ignore_index=True)
    del prov_chunks; gc.collect()

    for metric in METRICS_NET:
        mdf = prov_df[prov_df['metric'] == metric]
        grp = mdf.groupby(['source_agent','cloud_provider','region'])['value']
        p95 = grp.quantile(0.95).rename('p95')
        mad = grp.apply(lambda x: np.median(np.abs(x.values - np.median(x.values)))).rename('mad')
        stats = pd.concat([p95, mad], axis=1).reset_index()
        stats['metric'] = metric
        baseline_records.append(stats)
        del mdf; gc.collect()

    del prov_df; gc.collect()
    print(f"  ✅ {prov.upper()} listo")

baseline_df = pd.concat(baseline_records, ignore_index=True)
baseline_df['mad'] = baseline_df.apply(lambda r: max(r['mad'], MIN_MAD[r['metric']]), axis=1)

baseline_df.to_parquet(OUT_BASE, index=False)
print(f"\nBaseline guardado: {OUT_BASE}")
print(f"Shape: {baseline_df.shape}  (esperado ~3,105 = 1,035 rutas x 3 metricas)")
print(baseline_df.groupby('metric')[['p95','mad']].describe().round(4))


[AWS] 13 archivos...
  ✅ AWS listo
[AZURE] 13 archivos...
  ✅ AZURE listo
[GCP] 13 archivos...
  ✅ GCP listo

Baseline guardado: /srv/work/te_baseline_stats_v3.parquet
Shape: (3105, 6)  (esperado ~3,105 = 1,035 rutas x 3 metricas)
                    p95                                                  \
                  count    mean     std     min     25%     50%     75%   
metric                                                                    
network.jitter   1035.0  2.3384  2.8182  0.3878  0.5406  0.8089  2.9147   
network.latency  1035.0  0.1575  0.0817  0.0010  0.0983  0.1548  0.2006   
network.loss     1035.0  4.6274  8.8456  0.0000  0.0000  0.0200  6.0000   

                              mad                                         \
                      max   count    mean     std    min     25%     50%   
metric                                                                     
network.jitter    17.4490  1035.0  0.4655  0.7444  0.100  0.1224  0.1564   
network.latenc

In [4]:
baseline_df = pd.read_parquet(OUT_BASE)
baseline = {}
for row in baseline_df.itertuples():
    key = (row.source_agent, row.cloud_provider, row.region, row.metric)
    baseline[key] = {'p95': row.p95, 'mad': row.mad}

print(f"Baseline cargado: {len(baseline):,} entradas")

PROVIDER_MAP = {
    'aws':   [f for f in ARCHIVOS if '/te_aws_'   in f.lower()],
    'azure': [f for f in ARCHIVOS if '/te_azure_' in f.lower()],
    'gcp':   [f for f in ARCHIVOS if '/te_gcp_'   in f.lower()],
}

all_windows = []

for prov, files in PROVIDER_MAP.items():
    print(f"\n[{prov.upper()}] {len(files)} archivos...")
    for fi, f in enumerate(files):
        tbl = pq.read_table(f, columns=['_time','source_agent','cloud_provider','region','metric','value'])
        df  = tbl.to_pandas(); del tbl; gc.collect()
        df['_time'] = pd.to_datetime(df['_time'], utc=True)
        df = df[(df['_time'] >= BD_END) & (df['metric'].isin(METRICS_NET))].copy()
        if df.empty:
            del df; gc.collect()
            continue

        df['window'] = df['_time'].dt.floor('30min')

        for (sa, cp, reg, metric, win), grp in df.groupby(
                ['source_agent','cloud_provider','region','metric','window'], sort=False):
            bkey = (sa, cp, reg, metric)
            if bkey not in baseline:
                continue

            p95 = baseline[bkey]['p95']
            mad = baseline[bkey]['mad']
            t_baja  = p95 + K1_FINAL * mad
            t_media = max(p95 + K2_FINAL * mad, GUARDRAILS[metric]['media'])
            t_alta  = max(p95 + K3_FINAL * mad, GUARDRAILS[metric]['alta'])

            vals = grp['value'].values
            n    = len(vals)
            if n == 0:
                continue

            if   np.sum(vals > t_alta)  >= MIN_PERSISTENCE: lbl = 3
            elif np.sum(vals > t_media) >= MIN_PERSISTENCE: lbl = 2
            elif np.sum(vals > t_baja)  >= MIN_PERSISTENCE: lbl = 1
            else:                                            lbl = 0

            all_windows.append({
                'source_agent': sa, 'cloud_provider': cp, 'region': reg,
                'window': win, 'metric': metric, 'label': lbl, 'n_samples': n
            })

        del df; gc.collect()
        print(f"  [{fi+1}/{len(files)}] {os.path.basename(f)} — acum. {len(all_windows):,} filas")

    print(f"  ✅ {prov.upper()} listo")

windows_df = pd.DataFrame(all_windows)
del all_windows; gc.collect()

labeled_df = (
    windows_df
    .groupby(['source_agent','cloud_provider','region','window'])['label']
    .max()
    .reset_index()
    .rename(columns={'label': 'label_ventana'})
)

print(f"\nVentanas totales: {len(labeled_df):,}")
dist = labeled_df['label_ventana'].value_counts(normalize=True).sort_index() * 100
labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
for k, pct in dist.items():
    print(f"  {labels_map[k]:8s} ({k}): {pct:.1f}%")
print(f"\n  No-normal total: {dist[dist.index > 0].sum():.1f}%")


Baseline cargado: 3,105 entradas

[AWS] 13 archivos...
  [2/13] te_aws_raw_may_w2.parquet — acum. 406,777 filas
  [3/13] te_aws_raw_may_w3.parquet — acum. 814,720 filas
  [4/13] te_aws_raw_may_w4.parquet — acum. 1,215,544 filas
  [5/13] te_aws_raw_may_w5.parquet — acum. 1,448,822 filas
  [6/13] te_aws_raw_w1.parquet — acum. 1,854,630 filas
  [7/13] te_aws_raw_w2.parquet — acum. 2,259,839 filas
  [8/13] te_aws_raw_w3.parquet — acum. 2,665,934 filas
  [9/13] te_aws_raw_w4.parquet — acum. 3,072,824 filas
  [10/13] te_aws_raw_w5.parquet — acum. 3,247,782 filas
  [11/13] te_aws_raw_w6.parquet — acum. 3,655,857 filas
  [12/13] te_aws_raw_w7.parquet — acum. 4,064,054 filas
  [13/13] te_aws_raw_w8.parquet — acum. 4,413,954 filas
  ✅ AWS listo

[AZURE] 13 archivos...
  [2/13] te_azure_raw_may_w2.parquet — acum. 4,730,340 filas
  [3/13] te_azure_raw_may_w3.parquet — acum. 5,047,629 filas
  [4/13] te_azure_raw_may_w4.parquet — acum. 5,365,065 filas
  [5/13] te_azure_raw_may_w5.parquet — acum. 5,5

In [5]:

def assign_split(ts):
    train_start, train_end = SPLIT_DATES['train']
    val_start,   val_end   = SPLIT_DATES['val']
    test_start,  test_end  = SPLIT_DATES['test']
    if   train_start <= ts < train_end: return 'train'
    elif val_start   <= ts < val_end:   return 'val'
    elif test_start  <= ts < test_end:  return 'test'
    else:                               return 'other'

labeled_df['split'] = labeled_df['window'].apply(assign_split)

print("=== Distribucion por split ===")
for split in ['train', 'val', 'test', 'other']:
    sdf = labeled_df[labeled_df['split'] == split]
    if sdf.empty:
        print(f"\n{split.upper()}: sin datos")
        continue
    n = len(sdf)
    dist = sdf['label_ventana'].value_counts(normalize=True).sort_index() * 100
    labels_map = {0:'Normal', 1:'Baja', 2:'Media', 3:'Alta'}
    w_min, w_max = sdf['window'].min(), sdf['window'].max()
    print(f"\n{split.upper()} ({n:,} ventanas | {str(w_min)[:10]} -> {str(w_max)[:10]}):")
    for k in [0,1,2,3]:
        pct = dist.get(k, 0)
        bar = '█' * int(pct / 3)
        print(f"  {labels_map[k]:8s} ({k}): {pct:.1f}%  {bar}")
    no_norm = dist[dist.index > 0].sum()
    print(f"  No-normal total: {no_norm:.1f}%")

labeled_df.to_parquet(OUT_LABEL, index=False)
print(f"\n✅ Guardado: {OUT_LABEL}")
print(f"   Shape: {labeled_df.shape}")
print(f"   Columnas: {list(labeled_df.columns)}")
print(f"\n   Splits:")
print(labeled_df['split'].value_counts())


=== Distribucion por split ===

TRAIN (2,125,778 ventanas | 2026-05-08 -> 2026-06-19):
  Normal   (0): 89.4%  █████████████████████████████
  Baja     (1): 7.2%  ██
  Media    (2): 1.7%  
  Alta     (3): 1.7%  
  No-normal total: 10.6%

VAL (644,690 ventanas | 2026-06-20 -> 2026-07-02):
  Normal   (0): 85.5%  ████████████████████████████
  Baja     (1): 9.1%  ███
  Media    (2): 2.3%  
  Alta     (3): 3.1%  █
  No-normal total: 14.5%

TEST (993,449 ventanas | 2026-07-03 -> 2026-07-22):
  Normal   (0): 86.1%  ████████████████████████████
  Baja     (1): 9.4%  ███
  Media    (2): 2.2%  
  Alta     (3): 2.3%  
  No-normal total: 13.9%

OTHER: sin datos

✅ Guardado: /srv/work/te_labeled_ventanas_v3.parquet
   Shape: (3763917, 6)
   Columnas: ['source_agent', 'cloud_provider', 'region', 'window', 'label_ventana', 'split']

   Splits:
split
train    2125778
test      993449
val       644690
Name: count, dtype: int64


In [6]:
IN_LABEL = f'{SAVE_DIR}/te_labeled_ventanas_v3.parquet'

if not os.path.exists(IN_LABEL):
    print('SKIP — te_labeled_ventanas_v3.parquet no encontrado. Ejecutar celda 4 primero.')
else:
    labeled_df = pd.read_parquet(IN_LABEL)
    labeled_df['window'] = pd.to_datetime(labeled_df['window'], utc=True)
    labels_map = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}

    print(f'Archivo : {IN_LABEL}')
    print(f'Shape   : {labeled_df.shape}')
    print(f'Columnas: {list(labeled_df.columns)}')

    print(f'\n=== Ventanas por split ===')
    print(labeled_df['split'].value_counts())

    print(f'\n=== Distribución de clases por split ===')
    for split in ['train', 'val', 'test']:
        sdf = labeled_df[labeled_df['split'] == split]
        if sdf.empty:
            print(f'\n{split.upper()}: sin datos')
            continue
        n    = len(sdf)
        dist = sdf['label_ventana'].value_counts(normalize=True).sort_index() * 100
        w_min, w_max = sdf['window'].min(), sdf['window'].max()
        print(f'\n{split.upper()} ({n:,} ventanas | {str(w_min)[:10]} → {str(w_max)[:10]}):')
        for k in [0, 1, 2, 3]:
            pct = dist.get(k, 0.0)
            bar = '█' * int(pct / 3)
            print(f'  {labels_map[k]:8s} ({k}): {pct:5.1f}%  {bar}')
        no_norm = dist[dist.index > 0].sum()
        print(f'  No-normal total: {no_norm:.1f}%')


Archivo : /srv/work/te_labeled_ventanas_v3.parquet
Shape   : (3763917, 6)
Columnas: ['source_agent', 'cloud_provider', 'region', 'window', 'label_ventana', 'split']

=== Ventanas por split ===
split
train    2125778
test      993449
val       644690
Name: count, dtype: int64

=== Distribución de clases por split ===

TRAIN (2,125,778 ventanas | 2026-05-08 → 2026-06-19):
  Normal   (0):  89.4%  █████████████████████████████
  Baja     (1):   7.2%  ██
  Media    (2):   1.7%  
  Alta     (3):   1.7%  
  No-normal total: 10.6%

VAL (644,690 ventanas | 2026-06-20 → 2026-07-02):
  Normal   (0):  85.5%  ████████████████████████████
  Baja     (1):   9.1%  ███
  Media    (2):   2.3%  
  Alta     (3):   3.1%  █
  No-normal total: 14.5%

TEST (993,449 ventanas | 2026-07-03 → 2026-07-22):
  Normal   (0):  86.1%  ████████████████████████████
  Baja     (1):   9.4%  ███
  Media    (2):   2.2%  
  Alta     (3):   2.3%  
  No-normal total: 13.9%


In [7]:
IN_LABEL = f'{SAVE_DIR}/te_labeled_ventanas_v3.parquet'

if not os.path.exists(IN_LABEL):
    print('SKIP — te_labeled_ventanas_v3.parquet no encontrado. Ejecutar celda 4 primero.')
else:
    labeled_df = pd.read_parquet(IN_LABEL)
    labeled_df['window'] = pd.to_datetime(labeled_df['window'], utc=True)
    test_df = labeled_df[labeled_df['split'] == 'test'].copy()

    if test_df.empty:
        print('TEST sin datos en el archivo etiquetado.')
    else:
        test_df = test_df.sort_values('window').reset_index(drop=True)
        t_min, t_max = test_df['window'].min(), test_df['window'].max()
        expected = pd.date_range(t_min, t_max, freq='30min')
        n_routes = test_df[['source_agent','cloud_provider','region']].drop_duplicates().shape[0]

        print('=== Cobertura del test set ===')
        print(f'  Período     : {str(t_min)[:19]} → {str(t_max)[:19]}')
        print(f'  Días        : {(t_max - t_min).days + 1}')
        print(f'  Rutas       : {n_routes:,}')
        print(f'  Ventanas    : {len(test_df):,}')
        total_esp = len(expected) * n_routes
        print(f'  Esperadas   : {total_esp:,}  ({len(expected)} slots × {n_routes} rutas)')
        print(f'  Cobertura   : {100 * len(test_df) / total_esp:.1f}%')

        test_df['date'] = test_df['window'].dt.date
        by_day = test_df.groupby('date').size()
        print(f'\n  Ventanas por día (test):')
        for d, n in by_day.items():
            print(f'    {d}  {n:>10,}')

        labels_map = {0: 'Normal', 1: 'Baja', 2: 'Media', 3: 'Alta'}
        dist = test_df['label_ventana'].value_counts(normalize=True).sort_index() * 100
        print(f'\n  Distribución de clases en test:')
        for k in [0, 1, 2, 3]:
            pct = dist.get(k, 0.0)
            bar = '█' * int(pct / 3)
            print(f'    {labels_map[k]:8s} ({k}): {pct:5.1f}%  {bar}')


=== Cobertura del test set ===
  Período     : 2026-07-03 00:00:00 → 2026-07-22 23:30:00
  Días        : 20
  Rutas       : 1,035
  Ventanas    : 993,449
  Esperadas   : 993,600  (960 slots × 1035 rutas)
  Cobertura   : 100.0%

  Ventanas por día (test):
    2026-07-03      49,680
    2026-07-04      49,678
    2026-07-05      49,680
    2026-07-06      49,680
    2026-07-07      49,680
    2026-07-08      49,542
    2026-07-09      49,680
    2026-07-10      49,670
    2026-07-11      49,680
    2026-07-12      49,680
    2026-07-13      49,680
    2026-07-14      49,680
    2026-07-15      49,680
    2026-07-16      49,680
    2026-07-17      49,680
    2026-07-18      49,680
    2026-07-19      49,679
    2026-07-20      49,680
    2026-07-21      49,680
    2026-07-22      49,680

  Distribución de clases en test:
    Normal   (0):  86.1%  ████████████████████████████
    Baja     (1):   9.4%  ███
    Media    (2):   2.2%  
    Alta     (3):   2.3%  
